# LeetCode 2273 — Find Resultant Array After Removing Anagrams bằng Python

Link đề: https://leetcode.com/problems/find-resultant-array-after-removing-anagrams/ · Độ khó: **Easy** · Chủ đề: String, Sorting, Hash Table

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def removeAnagrams(self, words: List[str]) -> List[str]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **anagram** (hoán vị chữ cái) là gì và cách kiểm tra hai từ có phải anagram hay không bằng **sắp xếp** hoặc **đếm tần suất**.
2. Biết cách biến "một từ" thành **chữ ký (signature)** để hai anagram luôn có cùng chữ ký, rồi so sánh chữ ký thay vì so sánh từ.
3. Hiểu vì sao bài chỉ cần so sánh mỗi từ với **từ đứng ngay trước nó** và duyệt **một lượt từ trái sang phải** (không cần mô phỏng việc xóa).
4. Viết 5 cách: mô phỏng xóa thật sự, so với từ giữ lại gần nhất bằng `sorted`, bằng `Counter`, bằng mảng đếm 26 ô, và bằng `itertools.groupby` theo chữ ký.
5. Trace, test ngẫu nhiên đối chiếu, và tránh các lỗi: so sánh với từ gốc `words[0]` cố định, xóa phần tử khi đang duyệt danh sách, quên rằng `"aa"` và `"a"` không phải anagram.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `words` (mỗi từ chỉ gồm chữ thường). Ta thực hiện thao tác sau **nhiều lần cho tới khi không làm được nữa**:

- Chọn chỉ số `i` với `0 < i < len(words)` sao cho `words[i - 1]` và `words[i]` là **anagram** của nhau, rồi **xóa** `words[i]` khỏi mảng.

Trả về mảng `words` sau khi không còn thao tác nào. Có thể chứng minh kết quả **không phụ thuộc thứ tự chọn `i`**.

Hai từ là anagram nếu từ này có thể có được bằng cách sắp xếp lại **toàn bộ** chữ cái của từ kia, dùng mỗi chữ cái đúng một lần (ví dụ `"dacb"` và `"abdc"`).

| Ví dụ | words | Output | Giải thích |
|---|---|---|---|
| 1 | `["abba","baba","bbaa","cd","cd"]` | `["abba","cd"]` | `"baba"` là anagram của `"abba"` nên xóa; `"bbaa"` lại là anagram của `"abba"` nên xóa; `"cd"` thứ hai là anagram của `"cd"` thứ nhất nên xóa |
| 2 | `["a","b","c","d","e"]` | `["a","b","c","d","e"]` | không cặp liền kề nào là anagram |

Constraints: `1 <= len(words) <= 100`; `1 <= len(words[i]) <= 10`; `words[i]` chỉ gồm chữ cái thường.

In [1]:
examples = [
    (["abba", "baba", "bbaa", "cd", "cd"], ["abba", "cd"]),
    (["a", "b", "c", "d", "e"], ["a", "b", "c", "d", "e"]),
]
for words, expected in examples:
    pairs = [(words[i - 1], words[i], sorted(words[i - 1]) == sorted(words[i])) for i in range(1, len(words))]
    print(f"words={words}\n   các cặp liền kề (trước, sau, anagram?): {pairs}\n   mong đợi {expected}")

words=['abba', 'baba', 'bbaa', 'cd', 'cd']
   các cặp liền kề (trước, sau, anagram?): [('abba', 'baba', True), ('baba', 'bbaa', True), ('bbaa', 'cd', False), ('cd', 'cd', True)]
   mong đợi ['abba', 'cd']
words=['a', 'b', 'c', 'd', 'e']
   các cặp liền kề (trước, sau, anagram?): [('a', 'b', False), ('b', 'c', False), ('c', 'd', False), ('d', 'e', False)]
   mong đợi ['a', 'b', 'c', 'd', 'e']


## 3. Kiến thức nền

### 3.1. Anagram và `sorted(word)`

Hai từ là anagram khi chúng có **cùng nhiều chữ cái giống nhau** (cùng bội số từng chữ). Cách đơn giản nhất kiểm tra: sắp xếp các chữ cái của mỗi từ rồi so sánh. `sorted("baba")` trả về **danh sách** ký tự `['a', 'a', 'b', 'b']`; muốn ra chuỗi dùng `"".join(sorted(word))`.

In [2]:
print(sorted("baba"))
print("".join(sorted("baba")), "".join(sorted("abba")))
print(sorted("abba") == sorted("baba"))   # True: là anagram
print(sorted("aa") == sorted("a"))        # False: khác số lượng chữ cái
print(sorted("ab") == sorted("abc"))      # False: khác độ dài

['a', 'a', 'b', 'b']
aabb aabb
True
False
False


### 3.2. `collections.Counter` — đếm tần suất

`Counter(word)` đếm mỗi ký tự xuất hiện bao nhiêu lần, và hai `Counter` so sánh `==` được: bằng nhau khi mọi ký tự có cùng số đếm. Đây là cách kiểm tra anagram **không cần sắp xếp**, tốn thời gian tuyến tính theo độ dài từ.

In [3]:
from collections import Counter

print(Counter("abba"))
print(Counter("abba") == Counter("baba"))
print(Counter("aab") == Counter("abb"))   # cùng độ dài nhưng khác số đếm

Counter({'a': 2, 'b': 2})
True
False


### 3.3. Mảng đếm 26 ô với `ord`

Vì chỉ có chữ thường `a`–`z`, ta có thể đếm bằng mảng 26 phần tử: chữ cái `c` nằm ở ô `ord(c) - ord('a')`. Biến mảng thành `tuple` để dùng làm **khóa** (list không băm được, tuple thì được).

In [4]:
def count26(word):
    cnt = [0] * 26
    for ch in word:
        cnt[ord(ch) - ord('a')] += 1
    return tuple(cnt)

print(ord('a'), ord('d'), ord('d') - ord('a'))
print(count26("abba")[:5], count26("abba") == count26("baba"))
print(hash(count26("abba")) == hash(count26("baba")))

97 100 3
(2, 2, 0, 0, 0) True
True


### 3.4. `itertools.groupby` — nhóm các phần tử **liên tiếp** có cùng khóa

`groupby(iterable, key=f)` gom các phần tử **đứng kề nhau** có cùng `f(x)` thành một nhóm. Đúng với bài này: mỗi nhóm là một "đoạn" các từ liên tiếp cùng chữ ký, và ta chỉ giữ phần tử đầu mỗi nhóm.

In [5]:
from itertools import groupby

data = ["abba", "baba", "bbaa", "cd", "cd", "abba"]
for key, grp in groupby(data, key=lambda w: "".join(sorted(w))):
    print(key, list(grp))

aabb ['abba', 'baba', 'bbaa']
cd ['cd', 'cd']
aabb ['abba']


### 3.5. Tránh xóa phần tử khi đang duyệt danh sách

`del words[i]` trong lúc `for i in range(len(words))` làm danh sách ngắn đi và các chỉ số lệch. Cách an toàn là **xây danh sách kết quả mới** (hoặc duyệt từ cuối về đầu).

In [6]:
nums = [1, 2, 2, 3]
for i in range(len(nums) - 1, 0, -1):    # duyệt ngược: xóa không làm lệch các chỉ số chưa duyệt
    if nums[i] == nums[i - 1]:
        del nums[i]
print(nums)

[1, 2, 3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Thử bằng tay với `["abba","baba","bbaa","cd","cd"]`.

- Nhìn `abba` và `baba`: cùng chữ cái, xóa `baba`. Dãy còn `abba, bbaa, cd, cd`.
- Nhìn `abba` và `bbaa`: lại anagram, xóa `bbaa`. Còn `abba, cd, cd`.
- `abba` và `cd`: khác, giữ. `cd` và `cd`: anagram, xóa. Còn `abba, cd`.

Nhận xét quan trọng: sau khi xóa `baba`, từ đứng trước `bbaa` là **`abba`** — và `abba` là anagram của `baba` — nên "so với từ đứng ngay trước **trong mảng gốc**" hay "so với từ **giữ lại gần nhất**" cho cùng kết quả. Lý do: quan hệ "là anagram của" có tính **bắc cầu** (nếu A~B và B~C thì A~C).

Vậy ta không cần mô phỏng xóa: chỉ cần duyệt một lượt, giữ từ nào **khác anagram** với từ trước đó.

In [7]:
words = ["abba", "baba", "bbaa", "cd", "cd"]
keys = ["".join(sorted(w)) for w in words]
print(keys)                                   # chữ ký của từng từ
kept = [words[0]] + [words[i] for i in range(1, len(words)) if keys[i] != keys[i - 1]]
print(kept)

['aabb', 'aabb', 'aabb', 'cd', 'cd']
['abba', 'cd']


## 5. Cách 1 — Brute Force / baseline: mô phỏng việc xóa

**Ý tưởng:** làm đúng như đề: lặp "tìm `i` đầu tiên có `words[i-1]` anagram `words[i]`, xóa nó" cho tới khi không còn `i` nào. Mỗi vòng quét lại từ đầu.

**Độ phức tạp:** mỗi lần quét tốn `O(n * L log L)` (sắp xếp từ dài `L`), có thể phải quét tới `n` lần (mỗi lần xóa một từ) nên `O(n^2 * L log L)`. Với `n <= 100` vẫn rất nhỏ, nhưng **nút thắt** là ta quét lại cả mảng sau mỗi lần xóa trong khi phần đã kiểm tra không thay đổi.

In [8]:
from typing import List

class SolutionBrute:
    def removeAnagrams(self, words: List[str]) -> List[str]:
        words = list(words)                       # không sửa mảng của người gọi
        changed = True
        while changed:
            changed = False
            for i in range(1, len(words)):
                if sorted(words[i - 1]) == sorted(words[i]):
                    del words[i]                  # xóa rồi thoát vòng for để quét lại
                    changed = True
                    break
        return words

for w in (["abba", "baba", "bbaa", "cd", "cd"], ["a", "b", "c", "d", "e"]):
    print(w, "->", SolutionBrute().removeAnagrams(w))

['abba', 'baba', 'bbaa', 'cd', 'cd'] -> ['abba', 'cd']
['a', 'b', 'c', 'd', 'e'] -> ['a', 'b', 'c', 'd', 'e']


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** quan hệ anagram là quan hệ tương đương (phản xạ, đối xứng, bắc cầu), nên mỗi từ có một **chữ ký chuẩn** (từ đã sắp xếp, hoặc mảng đếm). Hai từ là anagram **khi và chỉ khi** chữ ký bằng nhau.

**Invariant:** khi xử lý từ thứ `i`, danh sách `res` đã chứa đúng kết quả cho `words[:i]`, và `res[-1]` có chữ ký bằng chữ ký của `words[i-1]`.

**Quy tắc:**

- Nếu `signature(words[i]) == signature(res[-1])` thì **bỏ** `words[i]`.
- Ngược lại thì **thêm** `words[i]` vào `res`.

Một lượt duyệt, mỗi từ tính chữ ký một lần: `O(n * L log L)` hoặc `O(n * L)` với mảng đếm.

## 7. Các cách tiếp theo

### Cách 2 — So với từ giữ lại gần nhất, dùng `sorted`

Giữ `res = [words[0]]`. Với mỗi từ `w` còn lại, nếu `sorted(w) != sorted(res[-1])` thì thêm vào `res`. Code ngắn nhất, sạch nhất.

In [9]:
from typing import List

class SolutionSorted:
    def removeAnagrams(self, words: List[str]) -> List[str]:
        res = [words[0]]
        for w in words[1:]:
            if sorted(w) != sorted(res[-1]):
                res.append(w)
        return res

for w in (["abba", "baba", "bbaa", "cd", "cd"], ["a", "b", "c", "d", "e"]):
    print(w, "->", SolutionSorted().removeAnagrams(w))

['abba', 'baba', 'bbaa', 'cd', 'cd'] -> ['abba', 'cd']
['a', 'b', 'c', 'd', 'e'] -> ['a', 'b', 'c', 'd', 'e']


**Độ phức tạp Cách 2:** Time `O(n * L log L)`, Space `O(n * L)` cho mảng kết quả (hoặc `O(L)` phụ nếu không tính kết quả).

### Cách 3 — Dùng `Counter` để so sánh tần suất

Ý tưởng như Cách 2 nhưng chữ ký là `Counter(word)`. Lưu ý: đếm lại `Counter(res[-1])` mỗi lần là lãng phí, nên ta **nhớ chữ ký của từ vừa giữ**.

In [10]:
from typing import List
from collections import Counter

class SolutionCounter:
    def removeAnagrams(self, words: List[str]) -> List[str]:
        res = [words[0]]
        last = Counter(words[0])                  # chữ ký của từ giữ gần nhất
        for w in words[1:]:
            cur = Counter(w)
            if cur != last:
                res.append(w)
                last = cur
        return res

for w in (["abba", "baba", "bbaa", "cd", "cd"], ["z", "zz", "z"]):
    print(w, "->", SolutionCounter().removeAnagrams(w))

['abba', 'baba', 'bbaa', 'cd', 'cd'] -> ['abba', 'cd']
['z', 'zz', 'z'] -> ['z', 'zz', 'z']


**Độ phức tạp Cách 3:** Time `O(n * L)` (đếm mỗi từ một lần, so sánh hai `Counter` tối đa 26 khóa), Space `O(n * L)` kết quả cộng `O(1)` bộ đếm.

### Cách 4 — Mảng đếm 26 ô (`tuple`) làm chữ ký

Giống Cách 3 nhưng chữ ký là tuple 26 số — không phụ thuộc cấu trúc `Counter`, tận dụng tốt khi bảng chữ cái nhỏ và cố định.

In [11]:
from typing import List

class SolutionCount26:
    @staticmethod
    def sig(word):
        cnt = [0] * 26
        for ch in word:
            cnt[ord(ch) - 97] += 1
        return tuple(cnt)

    def removeAnagrams(self, words: List[str]) -> List[str]:
        res, last = [], None
        for w in words:
            s = self.sig(w)
            if s != last:                          # last=None ở lần đầu nên từ đầu luôn được giữ
                res.append(w)
                last = s
        return res

for w in (["abba", "baba", "bbaa", "cd", "cd"], ["ab", "ba", "ab", "bc"]):
    print(w, "->", SolutionCount26().removeAnagrams(w))

['abba', 'baba', 'bbaa', 'cd', 'cd'] -> ['abba', 'cd']
['ab', 'ba', 'ab', 'bc'] -> ['ab', 'bc']


**Độ phức tạp Cách 4:** Time `O(n * (L + 26))`, Space `O(n * L)` kết quả.

### Cách 5 — `itertools.groupby` theo chữ ký

Mỗi **đoạn** các từ liên tiếp cùng chữ ký chỉ để lại từ đầu tiên. `groupby` làm đúng việc gom đoạn đó, và `next(group)` lấy phần tử đầu.

In [12]:
from typing import List
from itertools import groupby

class SolutionGroupby:
    def removeAnagrams(self, words: List[str]) -> List[str]:
        return [next(grp) for _, grp in groupby(words, key=lambda w: "".join(sorted(w)))]

for w in (["abba", "baba", "bbaa", "cd", "cd"], ["a", "b", "a", "a"]):
    print(w, "->", SolutionGroupby().removeAnagrams(w))

['abba', 'baba', 'bbaa', 'cd', 'cd'] -> ['abba', 'cd']
['a', 'b', 'a', 'a'] -> ['a', 'b', 'a']


**Độ phức tạp Cách 5:** Time `O(n * L log L)`, Space `O(n * L)`. Ngắn gọn nhất nhưng người mới dễ vấp: `next(grp)` phải được gọi **trước khi** `groupby` chuyển sang nhóm kế tiếp — list comprehension này đã làm đúng vậy.

## 8. TRACE từng bước

Trace Cách 2 với `words = ["abba","baba","bbaa","cd","cd"]`:

| `i` | `w` | `res[-1]` | `sorted(w)` vs `sorted(res[-1])` | hành động | `res` |
|---|---|---|---|---|---|
| khởi đầu | | | | giữ từ đầu | `["abba"]` |
| 1 | `baba` | `abba` | `aabb` = `aabb` | bỏ | `["abba"]` |
| 2 | `bbaa` | `abba` | `aabb` = `aabb` | bỏ | `["abba"]` |
| 3 | `cd` | `abba` | `cd` != `aabb` | giữ | `["abba","cd"]` |
| 4 | `cd` | `cd` | `cd` = `cd` | bỏ | `["abba","cd"]` |

In [13]:
def trace_sorted(words):
    res = [words[0]]
    print(f"khởi đầu: res={res}")
    for i in range(1, len(words)):
        w, prev = words[i], res[-1]
        sw, sp = "".join(sorted(w)), "".join(sorted(prev))
        keep = sw != sp
        print(f"  i={i}: w={w!r} prev={prev!r} sorted: {sw!r} vs {sp!r} -> {'giữ' if keep else 'bỏ'}")
        if keep:
            res.append(w)
        print(f"       res={res}")
    return res

print("kết quả:", trace_sorted(["abba", "baba", "bbaa", "cd", "cd"]))

khởi đầu: res=['abba']
  i=1: w='baba' prev='abba' sorted: 'aabb' vs 'aabb' -> bỏ
       res=['abba']
  i=2: w='bbaa' prev='abba' sorted: 'aabb' vs 'aabb' -> bỏ
       res=['abba']
  i=3: w='cd' prev='abba' sorted: 'cd' vs 'aabb' -> giữ
       res=['abba', 'cd']
  i=4: w='cd' prev='cd' sorted: 'cd' vs 'cd' -> bỏ
       res=['abba', 'cd']
kết quả: ['abba', 'cd']


In [14]:
def trace_brute(words):
    words = list(words)
    vong = 0
    while True:
        vong += 1
        idx = next((i for i in range(1, len(words)) if sorted(words[i - 1]) == sorted(words[i])), None)
        print(f"  vòng {vong}: words={words} -> xóa chỉ số {idx}")
        if idx is None:
            return words
        del words[idx]

print("kết quả:", trace_brute(["abba", "baba", "bbaa", "cd", "cd"]))

  vòng 1: words=['abba', 'baba', 'bbaa', 'cd', 'cd'] -> xóa chỉ số 1
  vòng 2: words=['abba', 'bbaa', 'cd', 'cd'] -> xóa chỉ số 1
  vòng 3: words=['abba', 'cd', 'cd'] -> xóa chỉ số 2
  vòng 4: words=['abba', 'cd'] -> xóa chỉ số None
kết quả: ['abba', 'cd']


## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def removeAnagrams(self, words: List[str]) -> List[str]:
        res = [words[0]]
        for w in words[1:]:
            if sorted(w) != sorted(res[-1]):
                res.append(w)
        return res

print(Solution().removeAnagrams(["abba", "baba", "bbaa", "cd", "cd"]))
print(Solution().removeAnagrams(["a", "b", "c", "d", "e"]))

['abba', 'cd']
['a', 'b', 'c', 'd', 'e']


## 10. Vì sao thuật toán đúng

- **Chữ ký quyết định anagram.** Hai từ có cùng nhiều chữ cái giống nhau từng loại thì sắp xếp ra cùng một chuỗi; ngược lại nếu khác thì không. Nên so chữ ký là so anagram.
- **Bắc cầu.** Nếu `A ~ B` và `B ~ C` thì `A ~ C`. Vì vậy dù đề nói "so với `words[i-1]` hiện tại" (sau khi đã xóa), từ đứng trước chính là từ giữ lại gần nhất, và chữ ký của nó **bằng** chữ ký của mọi từ đã bị xóa trước đó trong cùng đoạn.
- **Mỗi đoạn cùng chữ ký co lại còn một từ.** Mảng chia thành các đoạn liên tiếp cùng chữ ký; hai đoạn kề nhau luôn khác chữ ký (nếu bằng thì đã là một đoạn). Mỗi đoạn xóa dần đến khi còn từ đầu — và không thể xóa từ đầu đoạn vì từ trước nó khác chữ ký. Thuật toán giữ đúng từ đầu mỗi đoạn.
- **Kết quả không phụ thuộc thứ tự xóa** vì kết quả luôn là "từ đầu của mỗi đoạn".

## 11. Phân tích Time / Space Complexity

Gọi `n = len(words)`, `L` = độ dài lớn nhất của một từ (`<= 10`).

| Cách | Time | Space (phụ) |
|---|---|---|
| 1. Mô phỏng xóa | `O(n^2 * L log L)` | `O(n * L)` (bản sao) |
| 2. `sorted` | `O(n * L log L)` | `O(L)` |
| 3. `Counter` | `O(n * L)` | `O(1)` (tối đa 26 khóa) |
| 4. Mảng đếm 26 | `O(n * (L + 26))` | `O(26)` |
| 5. `groupby` | `O(n * L log L)` | `O(L)` |

Cộng thêm `O(n * L)` cho mảng kết quả. Với ràng buộc `n <= 100`, `L <= 10`, mọi cách đều chạy tức thì; chọn cách dễ đọc nhất.

## 12. Bộ test

Đối chiếu tất cả các cách trên nhiều input, gồm ví dụ đề, edge case, phản ví dụ và test ngẫu nhiên có `seed`.

In [16]:
import random

solutions = [SolutionBrute(), SolutionSorted(), SolutionCounter(), SolutionCount26(), SolutionGroupby(), Solution()]

fixed = [
    (["abba", "baba", "bbaa", "cd", "cd"], ["abba", "cd"]),
    (["a", "b", "c", "d", "e"], ["a", "b", "c", "d", "e"]),
    (["a"], ["a"]),                                   # chỉ một từ
    (["ab", "ba"], ["ab"]),                           # hai từ anagram
    (["aa", "a"], ["aa", "a"]),                       # khác độ dài: không là anagram
    (["ab", "ba", "ab"], ["ab"]),                     # bắc cầu qua từ đã xóa
    (["ab", "cd", "ba"], ["ab", "cd", "ba"]),         # anagram nhưng không liền kề
    (["z", "z", "z", "z"], ["z"]),                    # toàn bộ giống nhau
]
for words, expected in fixed:
    for s in solutions:
        assert s.removeAnagrams(list(words)) == expected, (type(s).__name__, words)
print("đã qua", len(fixed), "test cố định cho", len(solutions), "cách")

đã qua 8 test cố định cho 6 cách


In [17]:
random.seed(2273)
for _ in range(500):
    n = random.randint(1, 12)
    words = ["".join(random.choice("abc") for _ in range(random.randint(1, 4))) for _ in range(n)]
    ref = SolutionBrute().removeAnagrams(list(words))
    for s in solutions[1:]:
        assert s.removeAnagrams(list(words)) == ref, (type(s).__name__, words)
print("500 test ngẫu nhiên đối chiếu: OK")

500 test ngẫu nhiên đối chiếu: OK


In [18]:
# không làm thay đổi mảng đầu vào
data = ["abba", "baba", "cd"]
copy = list(data)
for s in solutions:
    s.removeAnagrams(data)
    assert data == copy
print("input không bị sửa: OK")

input không bị sửa: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: So sánh với từ đầu tiên `words[0]` mãi mãi

Sai: so mọi từ với `words[0]`. Ví dụ `["ab","cd","cd"]`: so `cd` với `ab` ta giữ cả hai `cd`. Phải so với **từ giữ lại gần nhất**.

In [19]:
def sai_so_voi_dau(words):
    res = [words[0]]
    for w in words[1:]:
        if sorted(w) != sorted(words[0]):      # SAI
            res.append(w)
    return res

def dung_so_voi_cuoi(words):
    res = [words[0]]
    for w in words[1:]:
        if sorted(w) != sorted(res[-1]):       # ĐÚNG
            res.append(w)
    return res

w = ["ab", "cd", "cd"]
print("sai :", sai_so_voi_dau(w))
print("đúng:", dung_so_voi_cuoi(w))

sai : ['ab', 'cd', 'cd']
đúng: ['ab', 'cd']


### Lỗi 2: Xóa phần tử trong lúc `for i in range(len(words))`

Sau `del words[i]`, độ dài giảm nhưng `range` đã được tính từ trước, nên sẽ gặp `IndexError` hoặc bỏ sót phần tử. Hãy xây danh sách mới.

In [20]:
def sai_xoa_khi_duyet(words):
    words = list(words)
    try:
        for i in range(1, len(words)):
            if sorted(words[i - 1]) == sorted(words[i]):
                del words[i]
    except IndexError as e:
        return "IndexError: " + str(e)
    return words

print("sai :", sai_xoa_khi_duyet(["ab", "ba", "ab", "ba"]))
print("đúng:", SolutionSorted().removeAnagrams(["ab", "ba", "ab", "ba"]))

sai : IndexError: list index out of range
đúng: ['ab']


### Lỗi 3: Chỉ so độ dài hoặc tổng mã ASCII thay vì so chữ ký đầy đủ

Hai từ cùng độ dài chưa chắc là anagram: `"ab"` và `"cd"`. Cùng tổng mã ASCII cũng chưa đủ: `"ad"` (97+100) và `"bc"` (98+99) cùng tổng 197 nhưng không là anagram.

In [21]:
def sai_tong(a, b):
    return sum(map(ord, a)) == sum(map(ord, b))     # SAI

def dung_sorted(a, b):
    return sorted(a) == sorted(b)

print("ad vs bc:", sai_tong("ad", "bc"), dung_sorted("ad", "bc"))
print("ab vs ba:", sai_tong("ab", "ba"), dung_sorted("ab", "ba"))

ad vs bc: True False
ab vs ba: True True


### Lỗi 4: Quên rằng `"aa"` và `"a"` không phải anagram

Nếu chỉ so **tập ký tự** (`set`) thì `set("aa") == set("a")` nên bị xem là anagram — sai, vì phải dùng đúng số lần.

In [22]:
print("set  :", set("aa") == set("a"))            # True nhưng SAI
print("sorted:", sorted("aa") == sorted("a"))       # False đúng
print(SolutionSorted().removeAnagrams(["aa", "a"]))

set  : True
sorted: False
['aa', 'a']


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy chữ "anagram" nghĩ ngay tới **chữ ký**: `sorted(word)` hoặc mảng đếm 26.
- Đề dùng từ **"liền kề"** + "xóa": thường chỉ cần **một lượt duyệt** giữ phần tử khác phần tử trước.
- "Gom đoạn liên tiếp giống nhau" → `itertools.groupby` (hoặc vòng for so `res[-1]`).
- Có tính bắc cầu thì không cần mô phỏng từng bước xóa.
- Khi ràng buộc nhỏ (`n <= 100`, `L <= 10`), ưu tiên code ngắn và đúng, không tối ưu sớm.

## 15. Biến thể của bài

- **Xóa anagram không liền kề:** giữ từ đầu của mỗi lớp chữ ký trên cả mảng (dùng `set` chữ ký đã thấy).
- **Đếm số nhóm anagram** (LeetCode 49 *Group Anagrams*).
- **Chỉ xóa nếu hai từ là anagram và khác chuỗi** (không xóa từ giống hệt).
- **Trả về chỉ số các từ bị xóa** thay vì mảng còn lại.
- **Chữ hoa/thường lẫn lộn hoặc Unicode:** chuẩn hóa `lower()` trước, và dùng `Counter` thay vì mảng 26.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | xóa thật, quét lại | `O(n^2 L log L)` | `O(nL)` | hiểu đề, đối chiếu |
| 2. `sorted` | so với `res[-1]` | `O(n L log L)` | `O(L)` | thi, phỏng vấn (ngắn nhất) |
| 3. `Counter` | chữ ký là bộ đếm | `O(n L)` | `O(1)` | từ dài, bảng chữ cái lớn |
| 4. Mảng 26 | chữ ký là tuple 26 | `O(n(L+26))` | `O(26)` | chữ thường cố định |
| 5. `groupby` | gom đoạn cùng chữ ký | `O(n L log L)` | `O(L)` | muốn code một dòng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, dự đoán `Solution().removeAnagrams(["ab", "ba", "cd", "dc", "ab"])`.

**Bài 2 (tính tay).** Với `["listen", "silent", "enlist", "google", "gogole"]`, danh sách còn lại là gì và có bao nhiêu từ bị xóa?

**Bài 3 (điền TODO).** Hoàn thành hàm `is_anagram(a, b)` bằng `Counter` (cell dưới chưa được gọi).

**Bài 4 (tự viết lại).** Viết `remove_all_anagrams(words)` giữ từ **đầu tiên** của mỗi lớp anagram trên **toàn mảng** (không cần liền kề).

**Bài 5 (mini challenge).** Viết `removed_indices(words)` trả về danh sách **chỉ số** các từ bị xóa theo bài gốc.

In [23]:
from collections import Counter

# TODO Bài 3: điền vào chỗ trống, KHÔNG gọi hàm trong cell này
def is_anagram_todo(a, b):
    # return Counter(...) == Counter(...)
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `ab`~`ba` nên xóa `ba`; `cd`~`dc` nên xóa `dc`; `ab` cuối so với `cd` thì khác. Bài 2: `listen/silent/enlist` cùng chữ ký, `google/gogole` cũng cùng chữ ký.
- **Mức 2 (vừa):** Bài 3: `Counter(a) == Counter(b)`. Bài 4: dùng `seen = set()` chứa chữ ký đã gặp; chỉ thêm từ nếu chữ ký chưa có.
- **Mức 3 (sâu):** Bài 5: duyệt như Cách 2 nhưng khi bỏ một từ thì ghi chỉ số `i` vào danh sách; nhớ `last` là chữ ký của từ giữ gần nhất.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
from collections import Counter

# Bài 1 và 2
assert Solution().removeAnagrams(["ab", "ba", "cd", "dc", "ab"]) == ["ab", "cd", "ab"]
r2 = Solution().removeAnagrams(["listen", "silent", "enlist", "google", "gogole"])
assert r2 == ["listen", "google"] and 5 - len(r2) == 3

# Bài 3
def is_anagram(a, b):
    return Counter(a) == Counter(b)

# Bài 4
def remove_all_anagrams(words):
    seen, res = set(), []
    for w in words:
        key = "".join(sorted(w))
        if key not in seen:
            seen.add(key)
            res.append(w)
    return res

# Bài 5
def removed_indices(words):
    removed, last = [], "".join(sorted(words[0]))
    for i in range(1, len(words)):
        key = "".join(sorted(words[i]))
        if key == last:
            removed.append(i)
        else:
            last = key
    return removed

assert is_anagram("abba", "baba") and not is_anagram("aa", "a")
assert remove_all_anagrams(["ab", "cd", "ba", "dc", "ef"]) == ["ab", "cd", "ef"]
assert removed_indices(["abba", "baba", "bbaa", "cd", "cd"]) == [1, 2, 4]
assert removed_indices(["a", "b", "c"]) == []
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (duyệt một lượt, so chữ ký với từ giữ gần nhất)

```python
class Solution:
    def removeAnagrams(self, words: List[str]) -> List[str]:
        res = [words[0]]
        for w in words[1:]:
            if sorted(w) != sorted(res[-1]):
                res.append(w)
        return res
```

**Checklist**

- [ ] Giữ từ đầu tiên, rồi mới duyệt từ chỉ số 1?
- [ ] So với `res[-1]` (từ giữ gần nhất), **không** so với `words[0]`?
- [ ] Không xóa phần tử khi đang duyệt chính danh sách đó?
- [ ] Anagram so bằng `sorted` hoặc `Counter` (không phải `set`, không phải tổng ASCII)?
- [ ] Đã thử `["a"]`, `["aa","a"]`, `["ab","ba","ab"]`?
- [ ] Signature đúng `removeAnagrams(self, words: List[str]) -> List[str]` và có `from typing import List`?